# JAX Model2 Runner: Layer-Routed Attention + Delayed Span LM

Trains the experimental `jax_model2/` architecture: normal/windowed attention refresh
points, far/medium/local delayed span families, and tokenwise layer attention over
previous block states. It reuses the original runner machinery: streaming packed
tokens, chunked tied-output cross entropy, split JIT compile, checkpointing, and
generation.


## 0. TPU / CUDA setup (skip on a local machine)

On a fresh Cloud TPU VM or Colab/Kaggle TPU runtime, uncomment the TPU line; on a
CUDA machine use the `jax[cuda12]` line instead (the repo's pinned `jax==0.4.34` +
`jax-metal` is for Apple Silicon only). Run once, then restart the kernel. Colab TPU
runtimes usually ship with a TPU-enabled `jax` already, in which case only the
clone + datasets/transformers/torch lines are needed. Torch is CPU-only here — it is
used solely for parameter initialization.

In [ ]:
# !git clone https://github.com/HyperRays/Hyperattn
# %cd Hyperattn
# !pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html   # TPU
# !pip install -U numpy "jax[cuda12]"   # NVIDIA GPU (recent jax needs numpy>=2.0 — restart kernel after)
# !pip install datasets transformers
# !pip install torch --index-url https://download.pytorch.org/whl/cpu

## 1. Imports and configuration

In [ ]:
import math
import pickle
import time
from dataclasses import asdict
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np

import jax_model2 as jax_model

n_devices = jax.device_count()
print("jax backend:", jax.default_backend())
print("jax devices:", n_devices, jax.devices())

seed = 1337

# -----------------------
# User-editable settings
# -----------------------

# Curriculum (hard cuts at step boundaries): grammar-heavy Simple English Wikipedia ->
# informational full English Wikipedia -> information-dense full-text arXiv. Fractions of
# max_iters; the last phase absorbs the rounding remainder.
curriculum_fractions = (0.2, 0.4, 0.4)
arxiv_dataset = "togethercomputer/RedPajama-Data-1T"  # full-text arXiv; fallback "ccdv/arxiv-summarization"
arxiv_config = "arxiv"                                 # ccdv fallback: "document"
arxiv_text_field = "text"                              # ccdv fallback: "article"
arxiv_trust_remote_code = True

# Fixed, neutral held-out validation set, intentionally OUTSIDE the curriculum so val_loss
# stays comparable across phases and tracks generalization rather than in-domain fit.
val_dataset_name = "HuggingFaceFW/fineweb-edu"
val_dataset_config = "sample-10BT"
val_text_field = "text"
tokenizer_name = "gpt2"
shuffle_buffer = 50_000
val_docs = 2_000

# Training shape. Keep divisible by n_devices.
batch_size = 32
block_size = 2048
assert batch_size % n_devices == 0

# Compute dtype: bf16 on TPU, fp32 elsewhere.
use_bfloat16 = jax.default_backend() == "tpu"
compute_dtype = jnp.bfloat16 if use_bfloat16 else jnp.float32
print("global batch:", batch_size, "| compute dtype:", compute_dtype.__name__)

# Model size.
n_embd = 384
n_head = 6
local_window = 256

# Coarse-to-fine span families. Each (width, lag) summarizes
# [position + 1 - lag - width, position + 1 - lag).
far_span_widths = (64, 128, 256, 512)
far_span_lags = (128, 256, 512, 1024)
mid_span_widths = (32, 64, 128, 256)
mid_span_lags = (32, 64, 128, 256)
local_span_widths = (2, 4, 8, 16, 32, 64)
local_span_lags = (0,)

# Stack: attention seed -> far delayed spans -> attention refresh -> medium spans ->
# attention refresh -> local spans. Tokenwise layer attention routes over all previous
# block states before every block after the first.
block_layout = jax_model.default_block_layout()
use_layer_attention = True
layer_attn_max_sources = 4     # embedding + 3 most-recent states. The routing bank [B, S, T, C]
                               # is applied (k/v proj + layernorm + softmax + concat) on EVERY
                               # block and the profile shows the model is HBM-bound, so bank width
                               # is the top throughput dial: 8->4 ~halves that traffic. None=all
                               # prior states -> O(depth) bank -> OOM. (raise back toward 8 if
                               # routing capacity matters more than speed.)
# Cross-layer HCA memory router: route each token over a compressed summary of the WHOLE
# vertical stack (streaming, learned softmax write) instead of the last few states. When True,
# layer_attn_max_sources is ignored. Cost ~ max_sources = layer_memory_slots. NEW mechanism --
# compare a short run against the windowed (use_memory_router=False) baseline before committing.
use_memory_router = True
layer_memory_slots = 4         # slot 0 = embedding (skip path) + 3 streaming memory slots
route_gate_init = 2.0          # sigmoid(2)=0.88 routed, 0.12 previous-state fallback.
ablation_interval = 0          # model2 ablation diagnostics are not wired yet.

# Optimization. LR uses a Warmup-Stable-Decay (WSD) schedule: linear warmup -> constant
# "stable" peak -> short 1-sqrt cooldown over the final lr_decay_frac of the run. The stable
# phase is horizon-free, so resuming/extending a run causes NO LR jump (cosine would). To
# extend training, raise max_iters and keep learning_rate/warmup_iters fixed; the cooldown
# just moves to the new end. With lr_decay_frac=0.2 the cooldown falls inside the arXiv stage.
max_iters = 10_000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4           # peak (stable) LR
min_lr_ratio = 0.1             # cooldown floor = min_lr_ratio * learning_rate
warmup_iters = 200
lr_decay_frac = 0.2            # fraction of max_iters spent cooling down at the end
weight_decay = 0.1
grad_clip = 1.0

# Optimizer: Muon on 2D weights, AdamW on everything else.
use_muon = True  # start with AdamW for first model2 compile; set True after a smoke run fits.
muon_momentum = 0.95
muon_ns_steps = 5
muon_rms_scale = 0.2
muon_lr_mult = 1.0
save_best_checkpoint = True
checkpoint_path = "best_jax_model2_layer_routed_lm.pkl"
latest_checkpoint_path = "latest_jax_model2_layer_routed_lm.pkl"

# Stochastic weight averaging (Hägele et al., NeurIPS 2024): a uniform running average of the
# params over the eval checkpoints from swa_start_frac*max_iters onward (the stable/cooldown
# tail). Free -- no extra training, just an averaged copy that usually beats the raw weights.
use_swa = True
swa_start_frac = 0.5
swa_checkpoint_path = "swa_jax_model2_layer_routed_lm.pkl"

# JAX backends.
attention_backend = "chunked"
span_backend = "fused"
remat_blocks = True
# Scan the fully-capped homogeneous span runs (mid/local) as one lax.scan body each instead of
# unrolling all blocks -> avoids the XLA compile-time host-RAM blowup on the deep stack. Requires
# layer_attn_max_sources set (the routing window must be fixed-size). Exact; parity-tested.
scan_span_runs = True

# Generation.
generate_tokens = 200
temperature = 0.6
top_k = 20
top_p = 0.9
repetition_penalty = 1.2
repetition_window = 128
frequency_penalty = 0.05
presence_penalty = 0.0
no_repeat_ngram_size = 4
min_new_tokens = 0
prompt = "The meaning of intelligence is"


## 2. Streaming packed-token dataset

Adapted from the original notebook's `StreamingPackedTokenDataset`, minus the torch
`DataLoader`: a plain generator that yields `(x, y)` NumPy int32 batches of shape
`(batch_size, block_size)`. The first `val_docs` documents are held out for validation;
the training stream skips them and shuffles with a buffer.

In [ ]:
from transformers import AutoTokenizer

from curriculum_data import (
    CurriculumLoader,
    build_val_iter,
    default_curriculum,
    phase_boundaries,
)

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
if tokenizer.eos_token_id is None:
    tokenizer.add_special_tokens({"eos_token": "<|endoftext|>"})
vocab_size = len(tokenizer)

curriculum = default_curriculum(
    max_iters,
    fractions=curriculum_fractions,
    arxiv_dataset=arxiv_dataset,
    arxiv_config=arxiv_config,
    arxiv_text_field=arxiv_text_field,
    arxiv_trust_remote_code=arxiv_trust_remote_code,
)
print("curriculum (hard cuts):")
for ph, end in zip(curriculum, phase_boundaries(curriculum)):
    cfgname = f"[{ph.dataset_config}]" if ph.dataset_config else ""
    print(f"  steps < {end:6d}  {ph.label:12s} {ph.dataset_name}{cfgname} field={ph.text_field}")

# Step-indexed train source: call train_curric.batch(step) once per step; it hard-switches
# datasets at the phase boundaries above and keeps only the active stream alive.
train_curric = CurriculumLoader(
    phases=curriculum,
    tokenizer=tokenizer,
    block_size=block_size,
    batch_size=batch_size,
    shuffle_buffer=shuffle_buffer,
    seed=seed,
)

val_iter = build_val_iter(
    dataset_name=val_dataset_name,
    dataset_config=val_dataset_config,
    text_field=val_text_field,
    tokenizer=tokenizer,
    block_size=block_size,
    batch_size=batch_size,
    shuffle_buffer=shuffle_buffer,
    seed=seed,
    val_docs=val_docs,
)

print("vocab_size:", vocab_size, "| eos:", tokenizer.eos_token, tokenizer.eos_token_id)


## 3. Model initialization

Initialize the experimental JAX-only model directly. Unlike the original runner,
this does not build a Torch model first.


In [ ]:
cfg = jax_model.LayerRoutedHGConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    local_window=local_window,
    dropout=0.0,
    block_layout=block_layout,
    far_span_widths=far_span_widths,
    far_span_lags=far_span_lags,
    mid_span_widths=mid_span_widths,
    mid_span_lags=mid_span_lags,
    local_span_widths=local_span_widths,
    local_span_lags=local_span_lags,
    use_layer_attention=use_layer_attention,
    layer_attn_max_sources=layer_attn_max_sources,
    route_gate_init=route_gate_init,
    use_memory_router=use_memory_router,
    layer_memory_slots=layer_memory_slots,
)
params = jax_model.init_params(jax.random.PRNGKey(seed), cfg)
print(f"parameters: {jax_model.count_parameters(params) / 1e6:.2f}M  ({len(block_layout)} blocks)")
print("layout:", block_layout)
print("far specs:", jax_model.span_specs_for_kind(cfg, "far_span"))
print("mid specs:", jax_model.span_specs_for_kind(cfg, "mid_span"))
print("local specs:", jax_model.span_specs_for_kind(cfg, "local_span"))


## 4. Device mesh and sharding

Standard data parallelism: a 1-D `"data"` mesh, parameters/optimizer state replicated,
batches sharded along the batch axis. With sharded inputs, `jax.jit` (GSPMD) partitions
the computation and all-reduces gradients without any change to the step function.
On a single device both helpers are no-ops.

In [ ]:
if n_devices > 1:
    from jax.sharding import Mesh, NamedSharding, PartitionSpec

    mesh = Mesh(np.asarray(jax.devices()), ("data",))
    _replicated = NamedSharding(mesh, PartitionSpec())
    _data_sharded = NamedSharding(mesh, PartitionSpec("data"))

    def replicate(tree):
        return jax.device_put(tree, _replicated)

    def shard_batch(a):
        return jax.device_put(jnp.asarray(a), _data_sharded)
else:
    def replicate(tree):
        return tree

    shard_batch = jnp.asarray

params = replicate(params)

## 5. Optimizer and train/eval steps

Self-contained AdamW (decoupled weight decay on all parameters, like the torch
notebook's `torch.optim.AdamW(model.parameters(), ...)`), cosine LR schedule with
warmup, and global-norm gradient clipping. The whole update is one jitted step with
donated params/optimizer buffers; `lr` is a traced argument so the schedule does not
trigger recompiles. When `use_bfloat16` is on, weights are cast to bf16 inside the
loss (gradients and the AdamW update stay fp32). The tied-output cross-entropy is
computed in vocab chunks, so the train step does not materialize `[batch, seq, vocab]`
logits.

In [ ]:
def get_lr(step):
    # Warmup-Stable-Decay (Hägele et al., NeurIPS 2024): warmup -> constant stable peak ->
    # 1-sqrt cooldown over the last lr_decay_frac of the run. The stable phase is horizon-free,
    # so resuming/extending the run produces no LR jump (cosine would). Only the cooldown
    # commits to a final horizon, so keep lr_decay_frac small unless this is the final run.
    min_lr = learning_rate * min_lr_ratio
    if step < warmup_iters:
        return learning_rate * step / max(1, warmup_iters)
    decay_iters = int(lr_decay_frac * max_iters)
    decay_start = max_iters - decay_iters
    if step < decay_start:
        return learning_rate
    progress = min(1.0, (step - decay_start) / max(1, decay_iters))
    return min_lr + (learning_rate - min_lr) * (1.0 - math.sqrt(progress))


def _is_muon_leaf(path, x):
    # Muon only on real 2D weight matrices; embedding / norms / biases / gates -> AdamW.
    if x.ndim != 2 or min(x.shape) == 1:
        return False
    return not any("token_embedding" in str(getattr(k, "key", k)) for k in path)


def newton_schulz(G, steps):
    # quintic Newton-Schulz: orthogonalize G (singular values -> 1), in bf16
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.astype(jnp.bfloat16)
    X = X / (jnp.linalg.norm(X) + 1e-7)
    transpose = G.shape[0] > G.shape[1]
    if transpose:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * (A @ A)) @ X
    if transpose:
        X = X.T
    return X.astype(G.dtype)


muon_mask = jax.tree_util.tree_map_with_path(_is_muon_leaf, params)
print(f"optimizer: Muon on {sum(1 for m in jax.tree.leaves(muon_mask) if m)} 2D matrices + AdamW on the rest"
      if use_muon else "optimizer: AdamW (use_muon=False)")

# Group Muon matrices by shape so Newton-Schulz runs as a few vmapped batches instead of
# ~180 unrolled per-matrix ops (the deep stack otherwise OOMs the XLA compiler on TPU).
muon_groups = {}
for _i, (_use, _shape) in enumerate(zip(jax.tree.leaves(muon_mask), [l.shape for l in jax.tree.leaves(params)])):
    if _use:
        muon_groups.setdefault(_shape, []).append(_i)


def opt_init(params):
    zeros = lambda: jax.tree.map(jnp.zeros_like, params)
    return {"s1": zeros(), "s2": zeros(), "step": jnp.zeros((), dtype=jnp.int32)}


def opt_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    grad_norm = jnp.sqrt(sum(jnp.sum(jnp.square(g)) for g in jax.tree.leaves(grads)))
    finite = jnp.isfinite(grad_norm)
    clip_scale = jnp.minimum(1.0, grad_clip / (grad_norm + 1e-12))
    grads = jax.tree.map(lambda g: g * clip_scale, grads)

    step = state["step"] + 1
    bc1 = 1 - beta1 ** step.astype(jnp.float32)
    bc2 = 1 - beta2 ** step.astype(jnp.float32)
    treedef = jax.tree_util.tree_structure(params)
    pl = jax.tree.leaves(params)
    gl = jax.tree.leaves(grads)
    s1l = jax.tree.leaves(state["s1"])
    s2l = jax.tree.leaves(state["s2"])
    new_p, new_s1, new_s2 = list(pl), list(s1l), list(s2l)
    muon_un = muon_pn = adam_un = adam_pn = 0.0

    used = set()
    if use_muon:  # Muon, batched per shape group: stack -> vmap(Newton-Schulz) -> scatter back
        ns = lambda gg: newton_schulz(gg, muon_ns_steps)
        for shp, idxs in muon_groups.items():
            P = jnp.stack([pl[i] for i in idxs])
            G = jnp.stack([gl[i] for i in idxs])
            buf = muon_momentum * jnp.stack([s1l[i] for i in idxs]) + G  # Nesterov momentum
            ortho = jax.vmap(ns)(G + muon_momentum * buf)
            scale = muon_lr_mult * muon_rms_scale * math.sqrt(max(shp))
            newP = P - lr * (scale * ortho + weight_decay * P)
            muon_un = muon_un + jnp.sum(jnp.square(newP - P))
            muon_pn = muon_pn + jnp.sum(jnp.square(P))
            for k, i in enumerate(idxs):
                new_p[i], new_s1[i] = newP[k], buf[k]  # s2 unused by Muon
                used.add(i)

    for i in range(len(pl)):  # AdamW for the remaining leaves
        if i in used:
            continue
        p, g, s1, s2 = pl[i], gl[i], s1l[i], s2l[i]
        m = beta1 * s1 + (1 - beta1) * g
        v = beta2 * s2 + (1 - beta2) * jnp.square(g)
        new = p - lr * ((m / bc1) / (jnp.sqrt(v / bc2) + eps) + weight_decay * p)
        adam_un = adam_un + jnp.sum(jnp.square(new - p))
        adam_pn = adam_pn + jnp.sum(jnp.square(p))
        new_p[i], new_s1[i], new_s2[i] = new, m, v

    muon_ratio = jnp.sqrt(muon_un) / (jnp.sqrt(muon_pn) + 1e-12)
    adam_ratio = jnp.sqrt(adam_un) / (jnp.sqrt(adam_pn) + 1e-12)

    keep = lambda new, old: jnp.where(finite, new, old)
    new_params = jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_p), params)
    new_state = {
        "s1": jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_s1), state["s1"]),
        "s2": jax.tree.map(keep, jax.tree_util.tree_unflatten(treedef, new_s2), state["s2"]),
        "step": keep(step, state["step"]),
    }
    return new_params, new_state, grad_norm, muon_ratio, adam_ratio


def loss_fn(p, x, y):
    if compute_dtype != jnp.float32:
        p = jax.tree.map(
            lambda a: a.astype(compute_dtype) if jnp.issubdtype(a.dtype, jnp.floating) else a, p
        )
    return jax_model.loss(
        p,
        x,
        y,
        cfg,
        attention_backend=attention_backend,
        span_backend=span_backend,
        remat_blocks=remat_blocks,
        scan_span_runs=scan_span_runs,
    )


# Split into two jitted programs so XLA compiles forward+backward and the optimizer
# SEPARATELY. The combined graph OOMs the compiler with Muon on (it tips the whole-
# program passes over a threshold that AdamW stays under); compiled apart, each fits.
@jax.jit
def compute_grads(params, x, y):
    return jax.value_and_grad(loss_fn)(params, x, y)


@partial(jax.jit, donate_argnums=(0, 1))
def apply_opt(params, opt_state, grads, lr):
    return opt_update(params, grads, opt_state, lr)


def train_step(params, opt_state, x, y, lr):
    loss, grads = compute_grads(params, x, y)
    params, opt_state, grad_norm, muon_ratio, adam_ratio = apply_opt(params, opt_state, grads, lr)
    return params, opt_state, loss, grad_norm, muon_ratio, adam_ratio


eval_step = jax.jit(loss_fn)


def estimate_loss(eval_params=None):
    p = params if eval_params is None else eval_params
    losses = []
    for _ in range(eval_iters):
        xb, yb = next(val_iter)
        losses.append(eval_step(p, shard_batch(xb), shard_batch(yb)))
    return float(jnp.mean(jnp.stack(losses)))


def print_gates(params):
    sig = lambda r: 1 / (1 + math.exp(-float(r)))
    span_g, route_g = [], []
    for i, block in enumerate(params["blocks"]):
        if "gate" in block:
            span_g.append((i, sig(np.asarray(block["gate"]))))
        if "route" in block and "gate" in block["route"]:
            route_g.append((i, sig(np.asarray(block["route"]["gate"]))))
    if span_g:
        v = [s for _, s in span_g]
        print(f"  span gates ({len(v)}): min {min(v):.3f}  mean {sum(v)/len(v):.3f}  max {max(v):.3f}")
    if route_g:
        v = [s for _, s in route_g[1:]] or [s for _, s in route_g]
        print(f"  route gates ({len(v)} active): min {min(v):.3f}  mean {sum(v)/len(v):.3f}  max {max(v):.3f}")


opt_state = replicate(opt_init(params))

## 5.5 Exact Compile Probe (split train_step)

`train_step` is now two jitted programs — `compute_grads` (forward+backward) and
`apply_opt` (Muon/AdamW optimizer) — compiled separately so the combined graph never
reaches the XLA backend at once. This cell probes each in isolation (lower + compile +
memory/cost analysis) so you can see which one is heavy and confirm both fit. Set
`PROBE_COMBINED = True` to also compile the old fused graph (expected to OOM with Muon).
Leave disabled during normal runs.

In [ ]:
RUN_TRAIN_STEP_COMPILE_PROBE = False
PROBE_COMBINED = False  # also compile the old fused forward+backward+optimizer graph


def summarize_compiled(name, compiled):
    try:
        print(f"  {name} memory_analysis:", compiled.memory_analysis())
    except Exception as e:
        print(f"  {name} memory_analysis unavailable:", repr(e))
    try:
        cost = compiled.cost_analysis()
        if isinstance(cost, list):
            cost = cost[0] if cost else {}
        for k in sorted(cost):
            if "bytes" in k.lower() or "flops" in k.lower() or "optimal" in k.lower():
                print(f"  {name} cost {k}: {cost[k]}")
    except Exception as e:
        print(f"  {name} cost_analysis unavailable:", repr(e))


def probe(name, fn, *args):
    t0 = time.time()
    lowered = fn.lower(*args)
    t1 = time.time()
    print(f"{name}: lowered in {t1 - t0:.1f}s")
    compiled = lowered.compile()
    print(f"{name}: compiled in {time.time() - t1:.1f}s")
    summarize_compiled(name, compiled)
    return compiled


if RUN_TRAIN_STEP_COMPILE_PROBE:
    xb_probe, yb_probe = train_curric.batch(0)
    xb_probe, yb_probe = shard_batch(xb_probe), shard_batch(yb_probe)
    print("probing split train_step with x:", xb_probe.shape, "y:", yb_probe.shape)

    # forward + backward (the same graph that compiles with use_muon=False)
    probe("compute_grads", compute_grads, params, xb_probe, yb_probe)

    # optimizer (Muon/AdamW), given real grads
    _, grads_probe = compute_grads(params, xb_probe, yb_probe)
    probe("apply_opt", apply_opt, params, opt_state, grads_probe, get_lr(1))

    if PROBE_COMBINED:
        @partial(jax.jit, donate_argnums=(0, 1))
        def _fused_train_step(params, opt_state, x, y, lr):
            loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
            return opt_update(params, grads, opt_state, lr)

        probe("FUSED train_step (expected to OOM with Muon)", _fused_train_step,
              params, opt_state, xb_probe, yb_probe, get_lr(1))
else:
    print("Compile probe disabled. Set RUN_TRAIN_STEP_COMPILE_PROBE = True to run it.")

## 6. Forward/backward smoke test

The first call compiles (slow); the second shows the steady-state step time.
(On Apple Metal a "donation is not implemented" warning is expected and harmless.)

In [ ]:
xb, yb = train_curric.batch(0)
xb, yb = shard_batch(xb), shard_batch(yb)
print("x:", xb.shape, "y:", yb.shape)

t0 = time.time()
params, opt_state, loss, grad_norm, muon_ratio, adam_ratio = train_step(params, opt_state, xb, yb, get_lr(1))
loss.block_until_ready()
print(f"first step (incl. compile): {time.time() - t0:.1f}s  loss={float(loss):.4f}")

xb, yb = train_curric.batch(1)
xb, yb = shard_batch(xb), shard_batch(yb)
t0 = time.time()
params, opt_state, loss, grad_norm, muon_ratio, adam_ratio = train_step(params, opt_state, xb, yb, get_lr(2))
loss.block_until_ready()
dt = time.time() - t0
print(f"steady-state step: {dt * 1000:.0f}ms  ({xb.size / dt:,.0f} tok/s)  "
      f"loss={float(loss):.4f}  grad_norm={float(grad_norm):.3f}  "
      f"update/param muon={float(muon_ratio):.2e} adam={float(adam_ratio):.2e}")

## 7. Resume (optional)

For long runs (e.g. 1M steps), the training loop writes a resumable `latest`
checkpoint every `eval_interval` — full optimizer moments + step + bookkeeping, not
just weights. To continue after a crash, set `resume = True` here and re-run the
notebook from the top. Leave it `False` to start fresh.

In [ ]:
# Resume support. The training loop writes a resumable "latest" checkpoint every
# eval_interval. To continue a run, set resume = True and re-run top to bottom.
resume = False


def save_checkpoint(path, step, val_loss):
    payload = {
        "params": jax.tree.map(np.asarray, jax.device_get(params)),
        "opt_state": jax.tree.map(np.asarray, jax.device_get(opt_state)),
        "config": asdict(cfg),
        "step": step,
        "best_val": best_val,
        "val_loss": val_loss,
        "train_loss_ema": loss_ema,
        "total_tokens": total_tokens,
        "tokenizer_name": tokenizer_name,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f)


def save_swa_checkpoint(path, step, val_loss):
    payload = {
        "params": swa_params,
        "swa_n": swa_n,
        "config": asdict(cfg),
        "step": step,
        "val_loss": val_loss,
        "tokenizer_name": tokenizer_name,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f)


start_step = 0
best_val = float("inf")
loss_ema = None
total_tokens = 0
swa_params = None
swa_n = 0

if resume:
    with open(latest_checkpoint_path, "rb") as f:
        ckpt = pickle.load(f)
    params = replicate(jax.tree.map(jnp.asarray, ckpt["params"]))
    opt_state = replicate(jax.tree.map(jnp.asarray, ckpt["opt_state"]))
    start_step = ckpt["step"] + 1
    best_val = ckpt.get("best_val", ckpt.get("val_loss", float("inf")))
    loss_ema = ckpt["train_loss_ema"]
    total_tokens = ckpt["total_tokens"]
    train_curric = CurriculumLoader(
        phases=curriculum,
        tokenizer=tokenizer,
        block_size=block_size,
        batch_size=batch_size,
        shuffle_buffer=shuffle_buffer,
        seed=seed + start_step,
    )
    if use_swa:
        try:
            with open(swa_checkpoint_path, "rb") as f:
                swa_ckpt = pickle.load(f)
            swa_params, swa_n = swa_ckpt["params"], swa_ckpt["swa_n"]
            print(f"resumed SWA average: n={swa_n}")
        except FileNotFoundError:
            print("no SWA checkpoint found; SWA average starts fresh")
    print(f"resumed: step {start_step}, best_val {best_val:.4f}, tokens {total_tokens:,}")
else:
    print("fresh start (resume=False)")


## 8. Training loop

In [ ]:
# best_val / loss_ema / total_tokens / start_step come from the resume cell above.
tokens_since_eval = 0
steps_since_eval = 0
clip_steps = 0
nonfinite_steps = 0
gnorm_ema = None
muon_ur_ema = None
adam_ur_ema = None
t0 = time.time()
abl_xb, abl_yb = next(val_iter)  # fixed batch -> comparable diagnostics across time

for step in range(start_step, max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        elapsed = time.time() - t0
        toks_per_sec = 0.0 if step == start_step else tokens_since_eval / max(elapsed, 1e-9)
        val_loss = estimate_loss()
        train_loss_str = "nan" if loss_ema is None else f"{loss_ema:.4f}"
        print(
            f"step {step:6d} | phase {train_curric.phase_label(step):11s} | train_ema {train_loss_str} | val {val_loss:.4f} | "
            f"lr {get_lr(step):.2e} | tok/s {toks_per_sec:,.0f} | "
            f"tokens {total_tokens:,} | elapsed {elapsed:.1f}s"
        )
        print_gates(params)
        clip_frac = clip_steps / max(1, steps_since_eval)
        gn_str = "nan" if gnorm_ema is None else f"{gnorm_ema:.3f}"
        mur = "nan" if muon_ur_ema is None else f"{muon_ur_ema:.2e}"
        aur = "nan" if adam_ur_ema is None else f"{adam_ur_ema:.2e}"
        print(f"  grad_norm_ema {gn_str} | update/param muon {mur} adam {aur} | clipped {clip_frac:.0%} | nonfinite {nonfinite_steps}")
        if ablation_interval and step % ablation_interval == 0:
            print("  model2 ablation diagnostics are not wired in this runner yet")
        if use_swa and step >= int(swa_start_frac * max_iters):
            # Fold current params into the uniform running average, then eval + checkpoint the
            # averaged weights. swa_n-th update keeps swa_params == mean of all folded snapshots.
            host_params = jax.tree.map(np.asarray, jax.device_get(params))
            if swa_params is None:
                swa_params, swa_n = host_params, 1
            else:
                swa_n += 1
                swa_params = jax.tree.map(lambda a, p: a + (p - a) / swa_n, swa_params, host_params)
            swa_val = estimate_loss(replicate(jax.tree.map(jnp.asarray, swa_params)))
            verdict = "better" if swa_val < val_loss else "worse"
            print(f"  swa (n={swa_n}) val {swa_val:.4f}  ({verdict} than raw {val_loss:.4f})")
            save_swa_checkpoint(swa_checkpoint_path, step, swa_val)

        t0 = time.time()
        tokens_since_eval = 0
        steps_since_eval = 0
        clip_steps = 0
        nonfinite_steps = 0

        # Always refresh the resumable "latest" checkpoint; keep the best separately.
        save_checkpoint(latest_checkpoint_path, step, val_loss)
        if save_best_checkpoint and val_loss < best_val:
            best_val = val_loss
            save_checkpoint(checkpoint_path, step, val_loss)
            print(f"saved best checkpoint to {checkpoint_path} with val={best_val:.4f}")

    if step == max_iters:
        break

    xb, yb = train_curric.batch(step)
    params, opt_state, loss, grad_norm, muon_ratio, adam_ratio = train_step(
        params, opt_state, shard_batch(xb), shard_batch(yb), get_lr(step)
    )

    # one host sync per step for all scalar metrics (cheaper on TPU than separate pulls)
    loss_value, gnorm, m_ur, a_ur = (float(v) for v in np.asarray(jnp.stack([loss, grad_norm, muon_ratio, adam_ratio])))
    if not (math.isfinite(loss_value) and math.isfinite(gnorm)):
        nonfinite_steps += 1  # the optimizer step was already skipped on-device; keep training
        continue
    loss_ema = loss_value if loss_ema is None else 0.99 * loss_ema + 0.01 * loss_value
    gnorm_ema = gnorm if gnorm_ema is None else 0.99 * gnorm_ema + 0.01 * gnorm
    muon_ur_ema = m_ur if muon_ur_ema is None else 0.99 * muon_ur_ema + 0.01 * m_ur
    adam_ur_ema = a_ur if adam_ur_ema is None else 0.99 * adam_ur_ema + 0.01 * a_ur
    clip_steps += int(gnorm > grad_clip)
    steps_since_eval += 1
    tokens_since_eval += xb.size
    total_tokens += xb.size

## 9. Generate text

All blocks are causal, so generation runs the model over a fixed-size buffer (one jit
compile) and reads the logits at the current position; the buffer slides once full.
Generation is fp32 and batch-1; on a multi-device mesh it simply runs replicated.

In [ ]:
# Optionally load a checkpoint before generation (best by val, or the most recent):
with open(checkpoint_path, "rb") as f:          # or latest_checkpoint_path
    ckpt = pickle.load(f)
params = replicate(jax.tree.map(jnp.asarray, ckpt["params"]))

gen_window = min(block_size, 512)


@jax.jit
def gen_logits(p, idx):
    return jax_model.forward(p, idx, cfg, attention_backend=attention_backend, span_backend=span_backend, remat_blocks=False, scan_span_runs=scan_span_runs)


def apply_no_repeat_ngram_np(logits, out, ngram_size):
    if ngram_size is None or ngram_size <= 0 or len(out) < ngram_size - 1:
        return logits
    prefix_len = ngram_size - 1
    prefix = tuple(out[-prefix_len:]) if prefix_len else tuple()
    banned = []
    for i in range(len(out) - ngram_size + 1):
        if tuple(out[i : i + prefix_len]) == prefix:
            banned.append(out[i + prefix_len])
    if banned:
        logits[np.asarray(banned, dtype=np.int64)] = -np.inf
    return logits


def apply_top_p_np(logits, top_p, min_tokens_to_keep=1):
    if top_p is None or top_p >= 1.0:
        return logits
    order = np.argsort(logits)[::-1]
    sorted_logits = logits[order]
    probs = np.exp(sorted_logits - np.max(sorted_logits))
    probs = probs / probs.sum()
    remove = np.cumsum(probs) > top_p
    remove[1:] = remove[:-1].copy()
    remove[:min_tokens_to_keep] = False
    remove[0] = False
    logits[order[remove]] = -np.inf
    return logits


def generate(params, prompt, max_new_tokens, temperature=0.8, top_k=50, top_p=None,
             repetition_penalty=1.2, repetition_window=128,
             frequency_penalty=0.0, presence_penalty=0.0, no_repeat_ngram_size=None,
             eos_token_id=None, min_new_tokens=0,
             seed=0):
    rng = np.random.default_rng(seed)
    ids = tokenizer.encode(prompt)
    out = list(ids)
    ids = ids[-gen_window:]
    buf = np.full((1, gen_window), tokenizer.eos_token_id, dtype=np.int32)
    buf[0, : len(ids)] = ids
    cur = len(ids)
    for step in range(max_new_tokens):
        logits = np.asarray(gen_logits(params, jnp.asarray(buf)))[0, cur - 1].astype(np.float64)
        recent_tokens = out[-repetition_window:] if repetition_window is not None else out
        if recent_tokens and (
            (repetition_penalty is not None and repetition_penalty != 1.0)
            or frequency_penalty != 0.0
            or presence_penalty != 0.0
        ):
            recent, counts = np.unique(np.asarray(recent_tokens, dtype=np.int64), return_counts=True)
            selected = logits[recent]
            if repetition_penalty is not None and repetition_penalty != 1.0:
                selected = np.where(selected > 0, selected / repetition_penalty, selected * repetition_penalty)
            if presence_penalty:
                selected = selected - presence_penalty
            if frequency_penalty:
                selected = selected - frequency_penalty * counts
            logits[recent] = selected
        logits = apply_no_repeat_ngram_np(logits, out, no_repeat_ngram_size)
        if eos_token_id is not None and step < min_new_tokens:
            logits[eos_token_id] = -np.inf
        logits = logits / temperature
        if top_k is not None:
            k = min(top_k, logits.shape[-1])
            cutoff = np.partition(logits, -k)[-k]
            logits = np.where(logits < cutoff, -np.inf, logits)
        logits = apply_top_p_np(logits, top_p)
        probs = np.exp(logits - logits.max()); probs /= probs.sum()
        nxt = int(rng.choice(len(probs), p=probs))
        out.append(nxt)
        if eos_token_id is not None and nxt == eos_token_id:
            break
        if cur == gen_window:
            buf[0, :-1] = buf[0, 1:]; cur -= 1
        buf[0, cur] = nxt; cur += 1
    return tokenizer.decode(out)


print(generate(
    params,
    prompt,
    generate_tokens,
    temperature=temperature,
    top_k=top_k,
    top_p=top_p,
    repetition_penalty=repetition_penalty,
    repetition_window=repetition_window,
    frequency_penalty=frequency_penalty,
    presence_penalty=presence_penalty,
    no_repeat_ngram_size=no_repeat_ngram_size,
    eos_token_id=tokenizer.eos_token_id,
    min_new_tokens=min_new_tokens,
    seed=seed,
))